# Projeto e mensagens-fonte — fase 2

Demonstração com **conversa fictícia** e **fixture local**. Aceitar uma interpretação na fila não grava o projeto. Uma segunda confirmação cria o projeto e preserva vínculos reais com as mensagens no SQLite. A anotação financeira abaixo documenta o que foi dito; **não é pagamento**.


In [ ]:
import sys, tempfile
from pathlib import Path

projeto = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(projeto/'src'))
from secretaria.banco import Banco, Contexto
from secretaria.extracao import analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
from secretaria.projetos_fontes import ProjetosFontes

temporario=tempfile.TemporaryDirectory()
caminho=Path(temporario.name)/'secretaria.sqlite3'
banco=Banco(caminho)
banco.preparar_demo()
ctx=Contexto('demo','ana')
corpus=banco.selecionar_corpus(ctx,'Demonstração fictícia')
banco.importar(ctx,corpus,projeto/'dados/ficticios/conversa_empresa.txt')
print('Versão SQLite:',banco.con.execute('PRAGMA user_version').fetchone()[0])


In [ ]:
resultado=analisar_snapshot(banco.snapshot_analise(ctx,corpus),criar_modelo_fixture(),'fixture_manual')
fila=FilaRevisao(banco)
ids=fila.registrar_resultado(ctx,resultado)['ids']
id_projeto,id_financeiro=ids[0],ids[1]
fila.decidir(ctx,id_projeto,1,'aceitar','Fonte do projeto conferida.')
fila.decidir(ctx,id_financeiro,1,'aceitar','Orçamento informado; não é pagamento.')
assert banco.con.execute('SELECT COUNT(*) FROM projetos').fetchone()[0]==0
print('Projetos após aceitar interpretações: 0')


In [ ]:
servico=ProjetosFontes(banco)
previa=servico.prever(ctx,id_projeto)
print('Prévia:',previa)
criacao=servico.confirmar(ctx,id_projeto,previa['versao'])
assert criacao['estado']=='confirmada'
print('Projeto confirmado:',criacao)
print('Fontes:',servico.fontes(ctx,criacao['anotacao_id']))


In [ ]:
previa_financeira=servico.prever(ctx,id_financeiro,projeto_id=criacao['projeto_id'])
anotacao=servico.confirmar(ctx,id_financeiro,previa_financeira['versao'],projeto_id=criacao['projeto_id'])
repeticao=servico.confirmar(ctx,id_financeiro,previa_financeira['versao'],projeto_id=criacao['projeto_id'])
assert repeticao['estado']=='ja_confirmada'
banco.fechar()
with Banco(caminho) as reaberto:
    fontes=ProjetosFontes(reaberto).fontes(ctx,anotacao['anotacao_id'])
    assert fontes
    print('Fontes após reabrir:',fontes)
    print('Anotações:',reaberto.con.execute('SELECT COUNT(*) FROM anotacoes_projeto').fetchone()[0])
temporario.cleanup()
